In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
dataset_path = '/content/drive/MyDrive/datasets'
print(os.listdir(dataset_path))

['data.yaml', 'test', 'valid', 'train']


In [ ]:
!pip install ultralytics

In [5]:
from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict
import random
import shutil
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
SRC_ROOT = Path('/content/drive/MyDrive/datasets') # 修改为正确的 Google Drive 路径
DST_ROOT = Path('/content/drive/MyDrive/datasets_subset') # 处理后的子集存放路径
TRAIN_RATIO = 0.2
VAL_RATIO = 0.3
seed = 42

NAMES = ['book', 'bottole', 'earphone', 'glass', 'headphone',
         'keyboard', 'laptop', 'mobile', 'mouse', 'pen', 'penstand']
NC = len(NAMES)

In [7]:
def sample_split(split:str,ratio:float,seed:int):
    """
    随机抽样图片，复制图片和标签到新目录
    Args:
        split: 数据集划分，'train'、'val'或'test'
        ratio: 抽样比例，0-1之间
        seed: 随机种子，确保可重复抽样
    """
    random.seed(seed)
    src_img_dir=SRC_ROOT/split/'images'     # 原始图像目录
    src_label_dir=SRC_ROOT/split/'labels'

    dst_img_dir=DST_ROOT/split/'images'     # 目标图像目录
    dst_label_dir=DST_ROOT/split/'labels'

    dst_img_dir.mkdir(parents=True, exist_ok=True)      # 创建目标图像目录
    dst_label_dir.mkdir(parents=True, exist_ok=True)

    images=list(src_img_dir.glob('*.*'))     # 获取所有图像文件
    if not images:
        print(f'{split} 目录下没有找到图像文件')
        return 0,0
    k=max(1,int(len(images)*ratio))     # 计算抽样数量，至少抽取1张
    sampled=random.sample(images,k)     # 随机抽样图像文件

    copied=0
    for img_path in sampled:
        lbl_path=src_label_dir /f'{img_path.stem}.txt'     # 对应的标签文件路径
        shutil.copy2(img_path,dst_img_dir/img_path.name)     # 复制图像文件
        if lbl_path.exists():
            shutil.copy2(lbl_path,dst_label_dir/lbl_path.name)     # 复制标签文件
        copied+=1

    print(f'[{split}]抽样{copied}/{len(images)}')
    return copied,len(images)


In [8]:
def build_subset_dataset():
    """
    构建小数据集目录，并生成 data.yaml。
    """
    data_yaml = {
        "path": str(DST_ROOT.absolute()),
        "train": "train/images",
        "val": "valid/images",
        "nc": NC,
        "names": NAMES,
    }

    DST_ROOT.mkdir(parents=True, exist_ok=True)
    with open(DST_ROOT / "data.yaml", "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

    print(f"已生成小数据集：{DST_ROOT}")

这一部分是用于分数据集并训练

In [ ]:
    if DST_ROOT.exists():
        print(f"删除旧数据集：{DST_ROOT}")
        shutil.rmtree(DST_ROOT)  # 递归删除整个文件夹
    sample_split("train", TRAIN_RATIO, seed)
    sample_split("valid", VAL_RATIO, seed)
    build_subset_dataset()
    model = YOLO("yolo26n.pt")
    model.train(
        data=str(DST_ROOT / "data.yaml"),
        epochs=50,
        imgsz=640,       # 输入图像大小
        verbose=False,       # 训练过程不输出详细日志
    )

这一部分是不同学习率的测试

In [9]:
import yaml
from pathlib import Path

DST_ROOT = Path('/content/drive/MyDrive/datasets_subset')

data_yaml = {
    "path": str(DST_ROOT),           # 数据集根目录
    "train": "train/images",         # 训练图片（相对于 path）
    "val": "valid/images",           # 验证图片（相对于 path）
    "nc": 11,
    "names": ['book', 'bottole', 'earphone', 'glass', 'headphone',
              'keyboard', 'laptop', 'mobile', 'mouse', 'pen', 'penstand']
}

# 写入文件
yaml_path = DST_ROOT / "data.yaml"
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print("✅ data.yaml 已配置")
print(f"\n配置内容:")
print("-" * 40)
print(yaml.dump(data_yaml, allow_unicode=True, default_flow_style=False))

# 验证路径
train_full = DST_ROOT / 'train' / 'images'
valid_full = DST_ROOT / 'valid' / 'images'

print(f"\n路径验证:")
print(f"训练图片路径: {train_full}")
print(f"  存在: {train_full.exists()}")
if train_full.exists():
    train_count = len(list(train_full.glob('*.*')))
    print(f"  图片数量: {train_count}")

print(f"\n验证图片路径: {valid_full}")
print(f"  存在: {valid_full.exists()}")
if valid_full.exists():
    valid_count = len(list(valid_full.glob('*.*')))
    print(f"  图片数量: {valid_count}")

✅ data.yaml 已配置

配置内容:
----------------------------------------
names:
- book
- bottole
- earphone
- glass
- headphone
- keyboard
- laptop
- mobile
- mouse
- pen
- penstand
nc: 11
path: /content/drive/MyDrive/datasets_subset
train: train/images
val: valid/images


路径验证:
训练图片路径: /content/drive/MyDrive/datasets_subset/train/images
  存在: True
  图片数量: 228

验证图片路径: /content/drive/MyDrive/datasets_subset/valid/images
  存在: True
  图片数量: 44


In [10]:
import pandas as pd

learning_rates=[0.001,0.005,0.01,0.02,0.05]
results_log = []
for lr in learning_rates:
  print(f"\n正在开始实验：初始学习率 lr0 = {lr}")

  # 加载模型
  model = YOLO("yolo26n.pt")

  # 训练模型
  model.train(
      data=str(DST_ROOT / "data.yaml"),
      epochs=20,
      imgsz=640,       # 输入图像大小
      batch=16,        # 批量大小
      lr0=lr,          # 初始学习率
      verbose=False,   # 训练过程不输出详细日志
      name=f"exp_lr_{lr}",
      exist_ok=True    # 如果文件夹已存在，允许覆盖
    )

  # 验证模型
  val_results = model.val(verbose=False)

  # 记录结果
  results_log.append({
      '实验': f'lr={lr}',
      'lr0': lr,
      'epochs': 20,
      'imgsz': 640,
      'batch': 16,
      'mAP50': val_results.box.map50,
      'mAP50-95': val_results.box.map,
      'P': val_results.box.p[0] if len(val_results.box.p) > 0 else 0,
      'R': val_results.box.r[0] if len(val_results.box.r) > 0 else 0,
    })

  print(f"实验结果 (lr={lr}): mAP50={val_results.box.map50:.3f}, mAP50-95={val_results.box.map:.3f}")

# 数据整理与保存
df = pd.DataFrame(results_log)
df.to_csv('exp_result_lr.csv', index=False)

print("\n" + "="*30)
print("所有实验已完成！结果已保存至 exp_result_lr.csv")
print(df[['实验', 'mAP50', 'mAP50-95']])


正在开始实验：初始学习率 lr0 = 0.001
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/datasets_subset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp_lr_0.001, nbs=64, nms=False, opset=None, optimize=Fal

下载文件

In [11]:
!zip -r /content/runs.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/val-5/ (stored 0%)
  adding: content/runs/detect/val-5/val_batch1_pred.jpg (deflated 16%)
  adding: content/runs/detect/val-5/BoxPR_curve.png (deflated 13%)
  adding: content/runs/detect/val-5/val_batch2_pred.jpg (deflated 12%)
  adding: content/runs/detect/val-5/val_batch2_labels.jpg (deflated 11%)
  adding: content/runs/detect/val-5/confusion_matrix.png (deflated 23%)
  adding: content/runs/detect/val-5/val_batch1_labels.jpg (deflated 16%)
  adding: content/runs/detect/val-5/val_batch0_labels.jpg (deflated 14%)
  adding: content/runs/detect/val-5/BoxP_curve.png (deflated 6%)
  adding: content/runs/detect/val-5/BoxR_curve.png (deflated 9%)
  adding: content/runs/detect/val-5/val_batch0_pred.jpg (deflated 15%)
  adding: content/runs/detect/val-5/confusion_matrix_normalized.png (deflated 19%)
  adding: content/runs/detect/val-5/BoxF1_curve.png (deflated 6%)
  adding: content/runs